# PET coincidence explorer

Choose **Runtime → Run all** in Colab, then use the controls. Two tracer deposits generate simulated annihilations inside a detector ring. Each event produces two nearly opposite photons. A detector pair in coincidence gives a **line of response (LOR)**.

The two images use the same events. One spreads each event along its line. The other uses its measured arrival-time difference to better localize the annihilation.

## What timing adds

Measure position $s$ from the midpoint of the line of response (LOR), positive toward detector B. If B detects its photon first, then $\Delta t=t_A-t_B$ is positive and the timing estimate is closer to B.

$$\hat s=\frac{c\,\Delta t}{2},\qquad
\mathrm{FWHM}_s=\frac{c\,\mathrm{FWHM}_{\Delta t}}{2},
\qquad c\approx0.300~\mathrm{mm/ps}.$$

FWHM means **full width at half maximum**. Here it generates an uncertainty along one line, in mm. The timing resolution control changes the uncertainty of the measured time difference. The event count control changes how many detector pairs enter both images.

In [3]:
#@title Launch PET explorer
from functools import lru_cache
from io import BytesIO

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize
from IPython.display import display

try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()

C = 0.299792458  # mm/ps
RING_RADIUS = 105  # mm
BODY_RADIUS = 80  # mm, displayed object boundary
CENTERS = np.array([[-28, -12], [27, 18]])
GRID = np.linspace(-80, 80, 161)
X, Y = np.meshgrid(GRID, GRID)
inside = X**2 + Y**2 <= BODY_RADIUS**2
rng = np.random.default_rng(313)
n_available = 400
events = CENTERS[rng.integers(0, len(CENTERS), n_available)] + rng.normal(
    scale=4, size=(n_available, 2))
angles = rng.uniform(0, 2*np.pi, n_available)
directions = np.column_stack((np.cos(angles), np.sin(angles)))
normals = np.column_stack((-np.sin(angles), np.cos(angles)))
offsets = (events * normals).sum(axis=1)
midpoints = offsets[:, None] * normals
half_chords = np.sqrt(RING_RADIUS**2 - offsets**2)
end_a = midpoints - half_chords[:, None] * directions
end_b = midpoints + half_chords[:, None] * directions
true_s = (events * directions).sum(axis=1)
timing_errors = rng.standard_normal(n_available)
FWHM_TO_SIGMA = 2*np.sqrt(2*np.log(2))


@lru_cache(maxsize=12)
def backprojections(count, timing_fwhm):
    """Sum unit-weight line kernels with and without TOF information."""
    sigma_s = C * timing_fwhm / (2 * FWHM_TO_SIGMA)
    estimated_s = true_s + sigma_s * timing_errors
    line_map, tof_map = np.zeros_like(X), np.zeros_like(X)
    for i in range(count):
        across = X*normals[i, 0] + Y*normals[i, 1] - offsets[i]
        along = X*directions[i, 0] + Y*directions[i, 1]
        line = np.exp(-0.5*(across/2)**2) * inside
        tof = line * np.exp(-0.5*((along-estimated_s[i])/sigma_s)**2)
        # Every accepted pair contributes the same total weight in each image.
        line_map += line / line.sum()
        tof_map += tof / tof.sum()
    return line_map/count, tof_map/count


def dashboard(count, timing_fwhm):
    line_map, tof_map = backprojections(count, timing_fwhm)
    fig, axes = plt.subplots(1, 3, figsize=(11.6, 3.9), layout="constrained")
    ax = axes[0]
    ax.add_patch(plt.Circle((0, 0), RING_RADIUS, fill=False, color="#536473"))
    ax.add_patch(plt.Circle((0, 0), BODY_RADIUS, fill=False, ls="--", ec="0.6"))
    for i in range(min(count, 12)):
        ax.plot([end_a[i, 0], end_b[i, 0]], [end_a[i, 1], end_b[i, 1]],
                lw=1, color="#157a80", alpha=.55)
    ax.scatter(CENTERS[:, 0], CENTERS[:, 1], marker="x", s=55,
               color="#bb4738", label="Deposit centers")
    ax.set(title=f"First {min(count, 12)} lines", xlim=(-112, 112),
           ylim=(-112, 112), aspect="equal", xlabel="x (mm)", ylabel="y (mm)")
    ax.legend(loc="lower center", fontsize=8)
    # Both images share one scale at each setting.
    norm = Normalize(0, max(line_map.max(), tof_map.max()))
    for ax, values, title in zip(axes[1:], (line_map, tof_map),
                                 ("Without timing", "With time of flight")):
        im = ax.imshow(values, extent=(-80.5, 80.5, -80.5, 80.5),
                       origin="lower", cmap="magma", norm=norm)
        ax.set(title=title, xlabel="x (mm)", ylabel="y (mm)")
    fig.colorbar(im, ax=axes[1:], shrink=.68, label="Relative weight per pixel")
    return fig


event_count = widgets.SelectionSlider(
    options=[8, 25, 80, 200, 400], value=80, description="Detected pairs",
    continuous_update=False, style={"description_width": "initial"},
    layout=widgets.Layout(width="360px", max_width="100%"))
timing = widgets.IntSlider(
    min=150, max=750, step=50, value=400, description="Timing FWHM (ps)",
    continuous_update=False, style={"description_width": "initial"},
    layout=widgets.Layout(width="380px", max_width="100%"))
readout = widgets.HTML()
plot = widgets.Image(format="png", layout=widgets.Layout(width="100%", max_width="1100px"))


def refresh(change=None):
    count, fwhm = event_count.value, timing.value
    fig = dashboard(count, fwhm)
    try:
        buffer = BytesIO()
        fig.savefig(buffer, format="png", dpi=115, facecolor="white")
        plot.value = buffer.getvalue()
    finally:
        plt.close(fig)
    readout.value = (
        f"<b>{count} detected pairs.</b> Along-line uncertainty: "
        f"<b>{C*fwhm/2:.1f} mm FWHM</b> at {fwhm} ps timing FWHM."
    )


event_count.observe(refresh, names="value")
timing.observe(refresh, names="value")
refresh()
display(widgets.VBox([
    widgets.Box([event_count, timing], layout=widgets.Layout(flex_flow="row wrap")),
    readout, plot
]))

## Investigate and explain

1. **One detector pair.** Set **Detected pairs**, the number of measured annihilations, to 8. Pick a line. How much does the pair tell you about where the annihilation occurred?
2. **More pairs.** Keep the timing resolution at 400 ps and increase from 8 to 400 pairs. Which features become easier to recognize? Which extra lines or streaks remain?
3. **Better photon timing.** Keep the pair count fixed and change from 600 to 200 ps. Use the readout to find the along-line FWHM in each case. How does the image change?

The images show sums of backprojected event weights. Each event contributes equal total weight, with the timing being used to redistribute the weight along the line. Both panels share a color scale at each setting. The scale auto-adjusts based on the controls. Compare feature shapes and separation across settings. The detector ring and events are idealized. The model assumes straight rays, continuous detector positions, and Gaussian timing errors, and omits attenuation, scatter, random pairs, and positron range.

[Further reading: Stanford Molecular Imaging Instrumentation Laboratory, TOF PET](https://med.stanford.edu/miil/research/tofdetector.html)

## From lines of response to an image

The opening explorer illustrates coincidence and timing. Next we will use many detector-pair measurements to reconstruct a slice, then examine what happens when photons are absorbed. 

### Uniform backprojection

Without timing, a single detected pair does not identify one location along its LOR. A simple image assigns equal weight to every candidate pixel along that line. In an idealized model with equal-size candidate pixels, event $i$ contributes

$$w_{ij}=\begin{cases}1/M_i,&\text{pixel }j\text{ is one of the }M_i\text{ candidates on line }i,\\0,&\text{otherwise,}\end{cases}
\qquad b_j=\sum_i w_{ij}.$$

Here $b_j$ is the sum of event weights at image pixel $j$. Each event contributes a total weight of one. Real implementations can also account for how much of a line crosses each pixel.

Uniform backprojection is a useful starting point: crossing lines reinforce one another. However, every event also puts weight at locations where it did not originate. Simply adding more events does not remove this systematic spreading.

### A three-pixel example

Suppose one LOR crosses three pixels with the same detection sensitivity. Uniform backprojection assigns the event weights $[1/3,1/3,1/3]$. Now suppose measurements from other directions suggest relative activities $[1,8,1]$ in these pixels. Given that estimate, a more reasonable allocation is

$$P(j\mid\text{event on line }i,\mathbf f)
=\frac{H_{ij}f_j}{\sum_m H_{im}f_m}
\quad\longrightarrow\quad[0.1,0.8,0.1].$$

$f_j$ describes the estimated activity in pixel $j$, and $H_{ij}$ describes the probability of detecting a decay in that pixel in measurement bin $i$. A **measurement bin** groups events detected by the same pair, or nearby pairs representing the same small range of lines. Equal $H_{ij}$ values cancel in this example.

This does not prove that this particular event came from the middle pixel. It allocates the event among possible origins using the current image and detector model. Repeating this calculation across all measurements leads to an iterative reconstruction.

**Check:** If the activity estimate were $[4,4,4]$, what weights would result? What if the middle pixel had half the detection sensitivity of the other two?

## Predicting detector counts from an image

For the next experiment, we group LORs by their angle $\theta$ and perpendicular distance $s$ from the slice center, as in the CT sinogram. Each bin contains a **number of detected coincidence events**, rather than an X-ray transmission or its logarithm. PET reconstruction estimates the distribution of the emitting tracer.

Our forward model is

$$y_i\sim\operatorname{Poisson}(\bar y_i),\qquad
\bar y_i=\sum_j H_{ij}f_j.$$

- $y_i$: measured coincidence count in bin $i$.
- $f_j$: fraction of the slice's reference activity in image pixel $j$ (dimensionless in this simulation).
- $H_{ij}$: expected counts in bin $i$ per unit activity fraction in pixel $j$, including exposure and detection probability.
- $\bar y_i$: predicted mean count in bin $i$.

The simulated object has two tracer deposits and a small uniform background. The known reference fractions sum to one. Reconstructed fractions are **not** forced to sum to one: their sum lets us see a loss of estimated activity when attenuation is ignored. Absolute activity in Bq would require a calibrated system model.

**Maximum likelihood estimation (MLE)** chooses an image that makes the measured counts most probable under this Poisson model. It does not mean setting every predicted count equal to its noisy measurement. Different measurement bins constrain the same image pixels.

One algorithm, **maximum likelihood expectation maximization (MLEM)**, starts with a positive uniform image and repeats:

1. Predict the counts from the current image.
2. Compare measured counts with predicted counts.
3. Redistribute the counts using the current image and update each pixel, accounting for its sensitivity.

For reference, its update is

$$f_j^{(k+1)}=\frac{f_j^{(k)}}{\sum_i H_{ij}}
\sum_i H_{ij}\frac{y_i}{\bar y_i^{(k)}},
\qquad \bar y_i^{(k)}=\sum_m H_{im}f_m^{(k)}.$$

The ratio $y_i/\bar y_i^{(k)}$ indicates whether the model predicts too few or too many counts. The denominator $\sum_i H_{ij}$ corrects for pixel sensitivity. Pixels must begin positive so that the multiplicative update can change them.

In [ ]:
#@title Launch reconstruction and attenuation experiments
from scipy import sparse

# A compact, explicitly discretized parallel-LOR model of one slice.
PET_N = 40
PET_DX = 2 * BODY_RADIUS / PET_N  # mm
pet_axis = -BODY_RADIUS + (np.arange(PET_N) + 0.5) * PET_DX
pet_x, pet_y = np.meshgrid(pet_axis, pet_axis)
pet_mask = pet_x**2 + pet_y**2 < BODY_RADIUS**2
pet_xy = np.column_stack((pet_x[pet_mask], pet_y[pet_mask]))
PET_ANGLES = np.linspace(0, np.pi, 60, endpoint=False)
PET_S = np.linspace(-BODY_RADIUS, BODY_RADIUS, 65)  # mm
PET_DS = PET_S[1] - PET_S[0]
pet_rows, pet_cols, pet_weights = [], [], []
for a, theta in enumerate(PET_ANGLES):
    s = pet_xy[:, 0] * np.cos(theta) + pet_xy[:, 1] * np.sin(theta)
    coordinate = (s - PET_S[0]) / PET_DS
    lower = np.floor(coordinate).astype(int)
    fraction = coordinate - lower
    for shift, weight in ((0, 1 - fraction), (1, fraction)):
        pet_rows.extend(a * len(PET_S) + lower + shift)
        pet_cols.extend(np.arange(len(pet_xy)))
        pet_weights.extend(weight / len(PET_ANGLES))
PET_P = sparse.csr_matrix((pet_weights, (pet_rows, pet_cols)),
                         shape=(len(PET_ANGLES) * len(PET_S), len(pet_xy)))
# Each column sums to one: every ideal detected pair falls into one LOR bin.
pet_truth = np.full(len(pet_xy), 0.015)
for center in CENTERS:
    pet_truth += np.exp(-np.sum((pet_xy - center)**2, axis=1) / (2 * 6**2))
pet_truth /= pet_truth.sum()
pet_path = np.tile(2 * np.sqrt(np.maximum(0, BODY_RADIUS**2 - PET_S**2)),
                   len(PET_ANGLES))  # mm through a circular attenuating body
PET_STEPS = 40


def pet_image(values):
    image = np.full((PET_N, PET_N), np.nan)
    image[pet_mask] = values
    return image


@lru_cache(maxsize=12)
def pet_data(expected_pairs, mu):
    transmission = np.exp(-mu * pet_path)
    matrix = PET_P.multiply((expected_pairs * transmission)[:, None]).tocsr()
    mean = np.asarray(matrix @ pet_truth).ravel()
    # Fixed seed for each setting; both reconstructions use exactly the same data.
    local_rng = np.random.default_rng(np.random.SeedSequence(
        [31309, int(expected_pairs), int(round(mu * 1_000_000))]))
    measured = local_rng.poisson(mean)
    return matrix, measured, mean


def pet_mlem(matrix, measured, steps=PET_STEPS):
    sensitivity = np.asarray(matrix.sum(axis=0)).ravel()
    estimate = np.full(matrix.shape[1], 1 / matrix.shape[1])
    history = [estimate.copy()]
    likelihood = []
    for k in range(steps + 1):
        predicted = np.asarray(matrix @ estimate).ravel()
        safe = np.maximum(predicted, 1e-15)
        likelihood.append(float(np.sum(measured * np.log(safe) - predicted)))
        if k < steps:
            ratio = np.divide(measured, predicted, out=np.zeros_like(predicted),
                              where=predicted > 0)
            estimate = estimate * np.asarray(matrix.T @ ratio).ravel() / sensitivity
            history.append(estimate.copy())
    return np.asarray(history), np.asarray(likelihood)


@lru_cache(maxsize=16)
def pet_reconstruction(expected_pairs, mu, correct):
    physical_matrix, measured, _ = pet_data(expected_pairs, mu)
    model = physical_matrix if correct else expected_pairs * PET_P
    return pet_mlem(model, measured)


def pet_png(fig, target):
    buffer = BytesIO()
    fig.savefig(buffer, format='png', dpi=115, bbox_inches='tight')
    plt.close(fig)
    target.value = buffer.getvalue()


def pet_map(ax, values, title, vmax):
    artist = ax.imshow(pet_image(values), origin='lower', extent=(-80, 80, -80, 80),
                       cmap='magma', vmin=0, vmax=vmax)
    ax.set(title=title, xlabel='x (mm)', ylabel='y (mm)')
    return artist


pet_count_control = widgets.SelectionSlider(
    options=[('2,000', 2000), ('20,000', 20000)], value=20000,
    description='Expected pairs:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='410px'), continuous_update=False)
pet_iteration = widgets.IntSlider(value=10, min=0, max=PET_STEPS,
    description='Iteration:', continuous_update=False)
pet_recon_readout = widgets.HTML()
pet_recon_plot = widgets.Image(format='png')


def refresh_pet_reconstruction(change=None):
    count, k = pet_count_control.value, pet_iteration.value
    matrix, measured, _ = pet_data(count, 0.0)
    history, likelihood = pet_reconstruction(count, 0.0, True)
    estimate = history[k]
    # Directly distribute each bin's measured counts equally among its candidates,
    # with interpolation weights from the geometric model; normalize total weight.
    row_sum = np.asarray(PET_P.sum(axis=1)).ravel()
    allocation = np.divide(measured, row_sum, out=np.zeros_like(row_sum), where=row_sum > 0)
    backprojection = np.asarray(PET_P.T @ allocation).ravel() / max(measured.sum(), 1)
    predicted = np.asarray(matrix @ estimate).reshape(len(PET_ANGLES), len(PET_S))
    counts = measured.reshape(predicted.shape)
    fig, axes = plt.subplots(2, 3, figsize=(12, 7.4), constrained_layout=True)
    vmax = max(pet_truth.max(), estimate.max(), backprojection.max())
    for ax, values, title in zip(axes[0], [pet_truth, backprojection, estimate],
        ['Known activity', 'Uniform backprojection', f'MLEM: iteration {k}']):
        image = pet_map(ax, values, title, vmax)
    fig.colorbar(image, ax=list(axes[0]), shrink=0.75, label='Activity fraction / pixel')
    count_max = max(counts.max(), predicted.max(), 1)
    for ax, values, title in zip(axes[1, :2], [counts, predicted],
                               ['Measured counts', 'Predicted mean counts']):
        im = ax.imshow(values, origin='lower', aspect='auto',
            extent=(PET_S[0]-PET_DS/2, PET_S[-1]+PET_DS/2, -1.5, 178.5),
            cmap='viridis', vmin=0, vmax=count_max)
        ax.set(title=title, xlabel='LOR offset s (mm)', ylabel='Angle θ (degrees)')
    fig.colorbar(im, ax=list(axes[1, :2]), shrink=0.75, label='Counts / bin')
    axes[1, 2].plot(np.arange(len(likelihood)), likelihood-likelihood[0], color='#245c88')
    axes[1, 2].axvline(k, color='#be5723', linestyle='--')
    axes[1, 2].set(title='Fit to the measured counts', xlabel='Iteration',
                   ylabel='Log likelihood gain from start')
    pet_png(fig, pet_recon_plot)
    error = np.linalg.norm(estimate-pet_truth) / np.linalg.norm(pet_truth)
    pet_recon_readout.value = (
        f'<b>{measured.sum():,} measured pairs</b> · '
        f'Reconstructed total activity fraction: {estimate.sum():.3f} · '
        f'Relative image error: {error:.2f}<br>'
        'Image panels share a scale; the scale can change with the controls. '
        'Iteration 0 is the initial uniform estimate. The curve shows all 40 iterations.')

pet_count_control.observe(refresh_pet_reconstruction, names='value')
pet_iteration.observe(refresh_pet_reconstruction, names='value')
refresh_pet_reconstruction()
display(widgets.VBox([widgets.HTML('<h3>Iterative PET reconstruction</h3>'),
    pet_count_control, pet_iteration, pet_recon_readout, pet_recon_plot]))

## Investigate reconstruction

1. At 20,000 expected pairs, move from iteration 0 to 1, 5, 10, and 40. How do the deposits change? Compare the reconstructed image with uniform backprojection.
2. Compare the measured and predicted sinograms. Why should a good reconstruction predict the underlying pattern without reproducing every fluctuation in the measured counts?
3. Repeat with 2,000 expected pairs. Does an increase in likelihood always mean an image closer to the known object? Use the relative image error and look for noise at later iterations.

The displayed error is $\|\mathbf f^{(k)}-\mathbf f_{\rm true}\|_2/\|\mathbf f_{\rm true}\|_2$. We can compute it because this is a simulation. In a patient, the true image is unknown. MLEM improves the likelihood, but later iterations can amplify noise. Stopping early or adding prior information can help control that noise.

The backprojection panel is a normalized sum of geometric event weights, not a quantitative activity estimate. MLEM instead uses the forward count model. The two image types have different purposes even though they use the same measured data.

## Attenuation of a coincidence pair

Both 511-keV photons must reach their detectors without being removed from the accepted coincidence. For an annihilation at position $\mathbf r$ on a straight LOR,

$$P_{\rm pair}=\exp\!\left[-\int_{\mathbf r}^{A}\mu(\ell)\,d\ell\right]
\exp\!\left[-\int_{\mathbf r}^{B}\mu(\ell)\,d\ell\right]
=\exp\!\left[-\int_A^B\mu(\ell)\,d\ell\right].$$

$\mu$ is the linear attenuation coefficient at **511 keV**, in $\mathrm{mm}^{-1}$. The integrals are over the attenuating body; the exponent is dimensionless. Moving the annihilation along the same LOR makes one photon path longer and the other shorter. Their total optical path stays the same.

For a uniform circular body of radius $R$ and a line at distance $s$ from its center,

$$L(s)=2\sqrt{R^2-s^2},\qquad a(s)=e^{-\mu L(s)},\qquad
H_{ij}^{\rm attenuated}=a_iH_{ij}^{\rm no\ attenuation}.$$

For example, $\mu=0.010~\mathrm{mm}^{-1}$ and a total tissue path of 100 mm give $P_{\rm pair}=e^{-1}\approx0.37$. The body in this simulation has $R=80$ mm, so its central LOR has a longer, 160-mm tissue path.

In the next activity, the two reconstructions use **exactly the same attenuated counts**. One model ignores attenuation, and the other includes the known attenuation map. The reference activity and exposure stay fixed when $\mu$ changes. More attenuation therefore means fewer measured pairs.

In [ ]:
#@title Launch attenuation experiment
pet_mu_control = widgets.SelectionSlider(
    options=[('0', 0.0), ('0.005', 0.005), ('0.010', 0.010), ('0.020', 0.020)],
    value=0.010, description='μ (1/mm):', continuous_update=False,
    layout=widgets.Layout(width='440px'))
pet_atten_count = widgets.SelectionSlider(
    options=[('2,000', 2000), ('20,000', 20000)], value=20000,
    description='Pairs before attenuation:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='470px'), continuous_update=False)
pet_atten_iteration = widgets.IntSlider(value=20, min=1, max=PET_STEPS,
    description='Iteration:', continuous_update=False)
pet_atten_readout = widgets.HTML()
pet_atten_plot = widgets.Image(format='png')


def refresh_pet_attenuation(change=None):
    count, mu, k = pet_atten_count.value, pet_mu_control.value, pet_atten_iteration.value
    _, measured, mean = pet_data(count, mu)
    uncorrected = pet_reconstruction(count, mu, False)[0][k]
    corrected = pet_reconstruction(count, mu, True)[0][k]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
    vmax = max(pet_truth.max(), uncorrected.max(), corrected.max())
    for ax, values, title in zip(axes, [pet_truth, uncorrected, corrected],
        ['Known activity', 'Model ignores attenuation', 'Model includes attenuation']):
        image = pet_map(ax, values, title, vmax)
    fig.colorbar(image, ax=list(axes), shrink=0.8, label='Activity fraction / pixel')
    pet_png(fig, pet_atten_plot)
    pet_atten_readout.value = (
        f'Expected pairs: <b>{mean.sum():,.0f}</b> after attenuation '
        f'({count:,} without attenuation); measured: <b>{measured.sum():,}</b>.<br>'
        f'Central-LOR survival probability: {np.exp(-mu * 2 * BODY_RADIUS):.3f}. '
        f'Total activity fraction: known <b>1.000</b>; '
        f'ignoring attenuation <b>{uncorrected.sum():.3f}</b>; '
        f'including attenuation <b>{corrected.sum():.3f}</b>.<br>'
        'All panels share a color scale. Reconstructed fractions are not renormalized.')

for control in (pet_mu_control, pet_atten_count, pet_atten_iteration):
    control.observe(refresh_pet_attenuation, names='value')
refresh_pet_attenuation()
display(widgets.VBox([widgets.HTML('<h3>Attenuation and estimated activity</h3>'),
    pet_mu_control, pet_atten_count, pet_atten_iteration,
    pet_atten_readout, pet_atten_plot]))

## Investigate attenuation

1. With $\mu=0$, should the two reconstructions match? Check them. Increase $\mu$ to $0.010~\mathrm{mm}^{-1}$ while keeping exposure and iteration fixed. What happens to detected counts and the estimated total activity?
2. Calculate the survival probability for a pair on the central LOR ($L=160$ mm) at $\mu=0.010~\mathrm{mm}^{-1}$. Compare it with the readout. Why is this not necessarily the survival fraction for all detected lines combined?
3. At $\mu=0.020~\mathrm{mm}^{-1}$, compare 2,000 and 20,000 pairs before attenuation. Does including the correct attenuation model remove all noise? Explain the difference between correcting an activity bias and recovering photons that were never detected.

An attenuation-aware model can correct systematic underestimation of activity. It cannot restore the counting information carried by lost events. In clinical PET, an attenuation map is commonly estimated using CT or another measurement; it is not known perfectly.

### Model limits and further reading

This second experiment uses a new **binned, non-TOF** data set with many more events than the opening explorer. It models a 2D slice, 60 equally sampled LOR angles, 2.5-mm offset bins, and 4-mm square image pixels. Each image pixel's activity is represented at its center and shared between adjacent offset bins by linear interpolation. It is a compact teaching model rather than a calibrated scanner model. The two deposits have Gaussian standard deviation 6 mm. The boundary is a uniform attenuating circle, and the reconstruction assumes its shape and coefficient are known.

The simulations use fixed random seeds. They omit scatter, random coincidences, positron range, detector penetration, dead time, and motion. The new reconstruction uses a geometric line model without timing; incorporating timing would add an along-line response to $H_{ij}$. Real scanners also use spatially varying detector sensitivity and more detailed physical models.

MLEM maximizes the Poisson log likelihood (up to a constant independent of the image),

$$\ell(\mathbf f)=\sum_i\left[y_i\ln\bar y_i-\bar y_i\right].$$

A zero measured count still contributes through $-\bar y_i$: predicting events where none were seen also affects the fit. The teaching model has no additive scatter or random-coincidence background term.

For the original emission-tomography MLEM formulation, see [Shepp and Vardi, “Maximum likelihood reconstruction for emission tomography” (1982)](https://doi.org/10.1109/TMI.1982.4307558).